In [32]:
# ============================================================
# INTEGRATION 4a: Install Great Expectations
# ============================================================
%pip install great-expectations==0.18.15 --quiet
print('Great Expectations installed')


StatementMeta(, 33293fc6-eae5-46b7-9578-a38409178160, 62, Finished, Available, Finished, False)


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Great Expectations installed



In [45]:
DISCORD_WEBHOOK_URL = "https://discord.com/api/webhooks/1505961532167688212/mr9aOWRaoFGbnBjLRMwOQ8LE5TOACtrvUhUdLCKq2RFdri2enU4XJb8vuU9dMai254e7"

def send_alert(title, message, success=False):
    color = 3066993 if success else 15158332  # green or red
    payload = {
        "embeds": [{
            "title": f"Data Quality: {title}",
            "description": message,
            "color": color
        }]
    }
    try:
        r = requests.post(DISCORD_WEBHOOK_URL, json=payload, timeout=10)
        print(f"Discord webhook: HTTP {r.status_code}")
    except Exception as e:
        print(f"Discord error: {e}")

StatementMeta(, 33293fc6-eae5-46b7-9578-a38409178160, 76, Finished, Available, Finished, False)

In [34]:

# INTEGRATION: Push weather data to InfluxDB

import requests
import json

INFLUX_URL = "https://us-east-1-1.aws.cloud2.influxdata.com"
INFLUX_TOKEN = "egxUhVYGR4pHjxlgDIxPlQMmM2KR-Xk5yJ23-mH5k40ud4C6BxLKAHiLRbfS7MQl3mnn-khhASoV3wey9gDUeQ=="
INFLUX_ORG = "FabricDataProject"
INFLUX_BUCKET = "nyc_weather"

# Read weather records from Bronze
import json
df_weather_json = spark.read.option('multiline','true').json(
    'abfss://FabricDataProject@onelake.dfs.fabric.microsoft.com/Bronze.Lakehouse/Files/nyc_weather_jan2024.json'
)
records = [row.asDict() for row in df_weather_json.collect()]

# Convert to InfluxDB line protocol
lines = []
for r in records:
    from datetime import datetime
    ts_ns = int(datetime.fromisoformat(r['timestamp']).timestamp() * 1e9)
    line = f"nyc_weather,location=NYC temperature_c={r['temperature_c']},precipitation_mm={r['precipitation_mm']},windspeed_kmh={r['windspeed_kmh']} {ts_ns}"
    lines.append(line)

# Write to InfluxDB
payload = "\n".join(lines)
response = requests.post(
    f"{INFLUX_URL}/api/v2/write?org={INFLUX_ORG}&bucket={INFLUX_BUCKET}&precision=ns",
    headers={"Authorization": f"Token {INFLUX_TOKEN}", "Content-Type": "text/plain"},
    data=payload.encode('utf-8')
)
print(f"InfluxDB write: HTTP {response.status_code}")

StatementMeta(, 33293fc6-eae5-46b7-9578-a38409178160, 65, Finished, Available, Finished, False)

InfluxDB write: HTTP 400


In [3]:
import pandas as pd
# Load data into pandas DataFrame from "/lakehouse/default/Files/yellow_tripdata_2024-01.parquet"
df = pd.read_parquet("/lakehouse/default/Files/yellow_tripdata_2024-01.parquet")
display(df)


StatementMeta(, 33293fc6-eae5-46b7-9578-a38409178160, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4b090a6e-fc13-4dc4-ab0c-4e3e7e627aa9)

In [ ]:
import requests

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"

response = requests.get(url)
with open("/lakehouse/default/Files/yellow_tripdata_2024-01.parquet", "wb") as f:
    f.write(response.content)

print(f"Downloaded {len(response.content)} bytes")

StatementMeta(, 56a4d0e9-8093-40af-b91a-944266e333fb, 18, Finished, Available, Finished, False)

Downloaded 49961641 bytes


In [22]:
df = spark.read.parquet("Files/yellow_tripdata_2024-01.parquet")
print(f"Rows: {df.count()}")
print(f"Columns: {len(df.columns)}")
df.printSchema()

StatementMeta(, 56a4d0e9-8093-40af-b91a-944266e333fb, 19, Finished, Available, Finished, False)

Rows: 2964624
Columns: 19
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [23]:
import requests

url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
response = requests.get(url)

with open("/lakehouse/default/Files/taxi_zone_lookup.csv", "wb") as f:
    f.write(response.content)

print("Done")

StatementMeta(, 56a4d0e9-8093-40af-b91a-944266e333fb, 20, Finished, Available, Finished, False)

Done


In [24]:
import pandas as pd
zones = pd.read_csv("/lakehouse/default/Files/taxi_zone_lookup.csv")
print(zones.head(10))

StatementMeta(, 56a4d0e9-8093-40af-b91a-944266e333fb, 21, Finished, Available, Finished, False)

   LocationID        Borough                     Zone service_zone
0           1            EWR           Newark Airport          EWR
1           2         Queens              Jamaica Bay    Boro Zone
2           3          Bronx  Allerton/Pelham Gardens    Boro Zone
3           4      Manhattan            Alphabet City  Yellow Zone
4           5  Staten Island            Arden Heights    Boro Zone
5           6  Staten Island  Arrochar/Fort Wadsworth    Boro Zone
6           7         Queens                  Astoria    Boro Zone
7           8         Queens             Astoria Park    Boro Zone
8           9         Queens               Auburndale    Boro Zone
9          10         Queens             Baisley Park    Boro Zone


In [ ]:
import requests
import json

OPENAQ_KEY = "3da7bd656fe1925a5e0d7a2293b0412e4bccdc36775659ff06a171ba32ebb810"

# v3 uses /v3/sensors/{sensor_id}/measurements or /v3/locations/{id}/latest
# First let's find NYC specific location IDs
url = "https://api.openaq.org/v3/locations"
params = {
    "coordinates": "40.7128,-74.0060",
    "radius": 25000,
    "limit": 10
}
headers = {"X-API-Key": OPENAQ_KEY}

response = requests.get(url, params=params, headers=headers)
print(f"Status: {response.status_code}")
data = response.json()
print(json.dumps(data, indent=2))

StatementMeta(, 56a4d0e9-8093-40af-b91a-944266e333fb, 22, Finished, Available, Finished, False)

Status: 200
{
  "meta": {
    "name": "openaq-api",
    "website": "/",
    "page": 1,
    "limit": 10,
    "found": ">10"
  },
  "results": [
    {
      "id": 384,
      "name": "CCNY",
      "locality": "New York-Northern New Jersey-Long Island",
      "timezone": "America/New_York",
      "country": {
        "id": 155,
        "code": "US",
        "name": "United States"
      },
      "owner": {
        "id": 4,
        "name": "Unknown Governmental Organization"
      },
      "provider": {
        "id": 119,
        "name": "AirNow"
      },
      "isMobile": false,
      "isMonitor": true,
      "instruments": [
        {
          "id": 2,
          "name": "Government Monitor"
        }
      ],
      "sensors": [
        {
          "id": 671,
          "name": "o3 ppm",
          "parameter": {
            "id": 10,
            "name": "o3",
            "units": "ppm",
            "displayName": "O\u2083"
          }
        },
        {
          "id": 673,
          "na

In [ ]:
import requests
import json

url = "https://api.worldbank.org/v2/country/USA/indicator/NY.GDP.MKTP.CD"
params = {
    "format": "json",
    "per_page": 20,
    "mrv": 10  # most recent 10 years
}

response = requests.get(url, params=params)
print(f"Status: {response.status_code}")
data = response.json()
print(json.dumps(data[1][:3], indent=2))

StatementMeta(, 56a4d0e9-8093-40af-b91a-944266e333fb, 23, Finished, Available, Finished, False)

Status: 200
[
  {
    "indicator": {
      "id": "NY.GDP.MKTP.CD",
      "value": "GDP (current US$)"
    },
    "country": {
      "id": "US",
      "value": "United States"
    },
    "countryiso3code": "USA",
    "date": "2024",
    "value": 28750956130731.2,
    "unit": "",
    "obs_status": "",
    "decimal": 0
  },
  {
    "indicator": {
      "id": "NY.GDP.MKTP.CD",
      "value": "GDP (current US$)"
    },
    "country": {
      "id": "US",
      "value": "United States"
    },
    "countryiso3code": "USA",
    "date": "2023",
    "value": 27292170793214.4,
    "unit": "",
    "obs_status": "",
    "decimal": 0
  },
  {
    "indicator": {
      "id": "NY.GDP.MKTP.CD",
      "value": "GDP (current US$)"
    },
    "country": {
      "id": "US",
      "value": "United States"
    },
    "countryiso3code": "USA",
    "date": "2022",
    "value": 25604848907611,
    "unit": "",
    "obs_status": "",
    "decimal": 0
  }
]


In [27]:
import requests
import json

OPENAQ_KEY = "3da7bd656fe1925a5e0d7a2293b0412e4bccdc36775659ff06a171ba32ebb810"

# Pull latest measurements from NYC location ID 384 (CCNY)
url = "https://api.openaq.org/v3/locations/384/sensors"
headers = {"X-API-Key": OPENAQ_KEY}

response = requests.get(url, headers=headers)
print(f"Status: {response.status_code}")
data = response.json()
print(json.dumps(data, indent=2))

StatementMeta(, 56a4d0e9-8093-40af-b91a-944266e333fb, 24, Finished, Available, Finished, False)

Status: 200
{
  "meta": {
    "name": "openaq-api",
    "website": "/",
    "page": 1,
    "limit": 100,
    "found": 2
  },
  "results": [
    {
      "id": 673,
      "name": "pm25 \u00b5g/m\u00b3",
      "parameter": {
        "id": 2,
        "name": "pm25",
        "units": "\u00b5g/m\u00b3",
        "displayName": "PM2.5"
      },
      "datetimeFirst": {
        "utc": "2016-03-12T09:00:00Z",
        "local": "2016-03-12T04:00:00-05:00"
      },
      "datetimeLast": {
        "utc": "2026-05-18T15:00:00Z",
        "local": "2026-05-18T11:00:00-04:00"
      },
      "coverage": {
        "expectedCount": 1,
        "expectedInterval": "01:00:00",
        "observedCount": 58225,
        "observedInterval": "58225:00:00",
        "percentComplete": 5822500.0,
        "percentCoverage": 5822500.0,
        "datetimeFrom": {
          "utc": "2016-03-12T09:00:00Z",
          "local": "2016-03-12T04:00:00-05:00"
        },
        "datetimeTo": {
          "utc": "2026-05-18T15:00:00Z

In [ ]:
import requests
import json

OPENAQ_KEY = "3da7bd656fe1925a5e0d7a2293b0412e4bccdc36775659ff06a171ba32ebb810"

# Pull PM2.5 measurements from CCNY sensor 673 for Jan 2024
url = "https://api.openaq.org/v3/sensors/673/measurements"
params = {
    "date_from": "2024-01-01T00:00:00Z",
    "date_to": "2024-01-31T23:59:59Z",
    "limit": 1000
}
headers = {"X-API-Key": OPENAQ_KEY}

response = requests.get(url, params=params, headers=headers)
print(f"Status: {response.status_code}")
data = response.json()
print(f"Records found: {data['meta']['found']}")

# Save to Bronze
with open("/lakehouse/default/Files/openaq_nyc_pm25.json", "w") as f:
    json.dump(data['results'], f)

print(f"Saved {len(data['results'])} records")
print(json.dumps(data['results'][0], indent=2))

StatementMeta(, 56a4d0e9-8093-40af-b91a-944266e333fb, 25, Finished, Available, Finished, False)

Status: 200
Records found: >1000
Saved 1000 records
{
  "value": 2.4,
  "flagInfo": {
    "hasFlags": false
  },
  "parameter": {
    "id": 2,
    "name": "pm25",
    "units": "\u00b5g/m\u00b3",
    "displayName": null
  },
  "period": {
    "label": "raw",
    "interval": "01:00:00",
    "datetimeFrom": {
      "utc": "2016-03-12T08:00:00Z",
      "local": "2016-03-12T03:00:00-05:00"
    },
    "datetimeTo": {
      "utc": "2016-03-12T09:00:00Z",
      "local": "2016-03-12T04:00:00-05:00"
    }
  },
  "coordinates": null,
  "summary": null,
  "coverage": {
    "expectedCount": 1,
    "expectedInterval": "01:00:00",
    "observedCount": 1,
    "observedInterval": "01:00:00",
    "percentComplete": 100.0,
    "percentCoverage": 100.0,
    "datetimeFrom": {
      "utc": "2016-03-12T08:00:00Z",
      "local": "2016-03-12T03:00:00-05:00"
    },
    "datetimeTo": {
      "utc": "2016-03-12T09:00:00Z",
      "local": "2016-03-12T04:00:00-05:00"
    }
  }
}


In [ ]:
import requests
import json

url = "https://api.worldbank.org/v2/country/USA/indicator/NY.GDP.MKTP.CD"
params = {
    "format": "json",
    "per_page": 20,
    "mrv": 10
}

response = requests.get(url, params=params)
records = response.json()[1]

# Save as JSON to Bronze
with open("/lakehouse/default/Files/usa_gdp.json", "w") as f:
    json.dump(records, f)

print(f"Saved {len(records)} GDP records")

StatementMeta(, 56a4d0e9-8093-40af-b91a-944266e333fb, 27, Finished, Available, Finished, False)

Saved 10 GDP records


In [31]:
import requests

url = "https://data-api.ecb.europa.eu/service/data/EXR/D.USD.EUR.SP00.A?format=csvdata"
response = requests.get(url)

with open("/lakehouse/default/Files/ecb_fx_usd_eur.csv", "wb") as f:
    f.write(response.content)

print(f"Downloaded {len(response.content)} bytes")

import pandas as pd
fx = pd.read_csv("/lakehouse/default/Files/ecb_fx_usd_eur.csv")
print(fx.shape)
print(fx.tail(3))

StatementMeta(, 56a4d0e9-8093-40af-b91a-944266e333fb, 28, Finished, Available, Finished, False)

Downloaded 1464666 bytes
(7069, 32)
                       KEY FREQ CURRENCY CURRENCY_DENOM EXR_TYPE EXR_SUFFIX  \
7066  EXR.D.USD.EUR.SP00.A    D      USD            EUR     SP00          A   
7067  EXR.D.USD.EUR.SP00.A    D      USD            EUR     SP00          A   
7068  EXR.D.USD.EUR.SP00.A    D      USD            EUR     SP00          A   

     TIME_PERIOD  OBS_VALUE OBS_STATUS OBS_CONF  ...  COMPILATION  COVERAGE  \
7066  2026-05-14     1.1702          A        F  ...          NaN       NaN   
7067  2026-05-15     1.1628          A        F  ...          NaN       NaN   
7068  2026-05-18     1.1648          A        F  ...          NaN       NaN   

     DECIMALS  NAT_TITLE SOURCE_AGENCY  SOURCE_PUB  \
7066        4        NaN           4F0         NaN   
7067        4        NaN           4F0         NaN   
7068        4        NaN           4F0         NaN   

                                           TITLE  \
7066  US dollar/Euro ECB reference exchange rate   
7067  US 

In [35]:
# ============================================================
# INTEGRATION 3: Weather Data via Open-Meteo API
# Fetch NYC hourly weather for January 2024
# Aligned with taxi data: Jan 2024
# ============================================================
import requests
import json
from datetime import datetime

LAT, LON = 40.7128, -74.0060

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": LAT,
    "longitude": LON,
    "start_date": "2024-01-01",
    "end_date": "2024-01-31",
    "hourly": "temperature_2m,precipitation,windspeed_10m,weathercode",
    "timezone": "America/New_York"
}

response = requests.get(url, params=params)
print(f"Weather API Status: {response.status_code}")
weather = response.json()

times = weather['hourly']['time']
temps = weather['hourly']['temperature_2m']
precip = weather['hourly']['precipitation']
wind = weather['hourly']['windspeed_10m']
wcode = weather['hourly']['weathercode']

records = [
    {
        'timestamp': t,
        'temperature_c': temp,
        'precipitation_mm': prec,
        'windspeed_kmh': ws,
        'weathercode': wc,
        'measurement': 'nyc_weather',
        'location': 'NYC'
    }
    for t, temp, prec, ws, wc in zip(times, temps, precip, wind, wcode)
]

with open('/lakehouse/default/Files/nyc_weather_jan2024.json', 'w') as f:
    json.dump(records, f, indent=2)

print(f"Saved {len(records)} hourly weather records for Jan 2024")
print(f"Date range: {times[0]} to {times[-1]}")
print(json.dumps(records[0], indent=2))

# InfluxDB line protocol (for Grafana integration)
print('\n--- InfluxDB Line Protocol sample (first 3 rows) ---')
for r in records[:3]:
    ts_ns = int(datetime.fromisoformat(r['timestamp']).timestamp() * 1e9)
    print(f"nyc_weather,location=NYC temperature_c={r['temperature_c']},precipitation_mm={r['precipitation_mm']},windspeed_kmh={r['windspeed_kmh']} {ts_ns}")


StatementMeta(, 33293fc6-eae5-46b7-9578-a38409178160, 66, Finished, Available, Finished, False)

Weather API Status: 200
Saved 744 hourly weather records for Jan 2024
Date range: 2024-01-01T00:00 to 2024-01-31T23:00
{
  "timestamp": "2024-01-01T00:00",
  "temperature_c": 1.9,
  "precipitation_mm": 0.0,
  "windspeed_kmh": 6.3,
  "weathercode": 3,
  "measurement": "nyc_weather",
  "location": "NYC"
}

--- InfluxDB Line Protocol sample (first 3 rows) ---
nyc_weather,location=NYC temperature_c=1.9,precipitation_mm=0.0,windspeed_kmh=6.3 1704067200000000000
nyc_weather,location=NYC temperature_c=1.8,precipitation_mm=0.0,windspeed_kmh=8.9 1704070800000000000
nyc_weather,location=NYC temperature_c=2.8,precipitation_mm=0.0,windspeed_kmh=11.4 1704074400000000000


In [36]:
# ============================================================
# Weather → Silver Delta table (time-series, Grafana-ready)
# ============================================================
from pyspark.sql import functions as F

BRONZE = 'abfss://FabricDataProject@onelake.dfs.fabric.microsoft.com/Bronze.Lakehouse/Files'

df_weather_raw = spark.read.option('multiline', 'true').json(f'{BRONZE}/nyc_weather_jan2024.json')

df_weather = df_weather_raw.select(
    F.to_timestamp('timestamp').alias('datetime'),
    F.to_date('timestamp').alias('date'),
    F.hour(F.to_timestamp('timestamp')).alias('hour'),
    F.col('temperature_c').cast('double'),
    F.col('precipitation_mm').cast('double'),
    F.col('windspeed_kmh').cast('double'),
    F.col('weathercode').cast('integer'),
    F.col('location'),
    F.col('measurement')
).filter(
    (F.col('date') >= '2024-01-01') & (F.col('date') <= '2024-01-31')
)

df_weather.write.format('delta').mode('overwrite').saveAsTable('silver_weather')
print(f'silver_weather: {df_weather.count()} hourly records')
df_weather.show(5, truncate=False)


print('\nGrafana: Use silver_weather table with GoldWarehouse SQL endpoint')
print('Query: SELECT datetime, temperature_c, precipitation_mm, windspeed_kmh FROM silver_weather ORDER BY datetime')


StatementMeta(, 33293fc6-eae5-46b7-9578-a38409178160, 67, Finished, Available, Finished, False)

silver_weather: 744 hourly records
+-------------------+----------+----+-------------+----------------+-------------+-----------+--------+-----------+
|datetime           |date      |hour|temperature_c|precipitation_mm|windspeed_kmh|weathercode|location|measurement|
+-------------------+----------+----+-------------+----------------+-------------+-----------+--------+-----------+
|2024-01-01 00:00:00|2024-01-01|0   |1.9          |0.0             |6.3          |3          |NYC     |nyc_weather|
|2024-01-01 01:00:00|2024-01-01|1   |1.8          |0.0             |8.9          |3          |NYC     |nyc_weather|
|2024-01-01 02:00:00|2024-01-01|2   |2.8          |0.0             |11.4         |3          |NYC     |nyc_weather|
|2024-01-01 03:00:00|2024-01-01|3   |2.9          |0.0             |9.7          |3          |NYC     |nyc_weather|
|2024-01-01 04:00:00|2024-01-01|4   |2.8          |0.0             |8.1          |2          |NYC     |nyc_weather|
+-------------------+----------+----+

In [44]:
# ============================================================
# Correlation Analysis - Taxi vs Weather (Jan 2024)
# Both datasets cover January 2024 hourly data
# ============================================================
from pyspark.sql import functions as F

SILVER = 'abfss://FabricDataProject@onelake.dfs.fabric.microsoft.com/Silver.Lakehouse/Tables/dbo'
BRONZE = 'abfss://FabricDataProject@onelake.dfs.fabric.microsoft.com/Bronze.Lakehouse/Tables/dbo'

# Load silver_taxi
df_taxi = spark.read.format('delta').load(f'{SILVER}/silver_taxi').select(
    'pickup_date', 'pickup_hour', 'trip_distance', 'fare_amount', 'trip_duration_min'
).filter(
    (F.col('pickup_date') >= '2024-01-01') & (F.col('pickup_date') <= '2024-01-31')
)

# Aggregate taxi by date + hour
df_taxi_hourly = df_taxi.groupBy('pickup_date', 'pickup_hour').agg(
    F.count('*').alias('trip_count'),
    F.avg('trip_distance').alias('avg_distance_miles'),
    F.avg('fare_amount').alias('avg_fare_usd'),
    F.avg('trip_duration_min').alias('avg_duration_min')
)

# Load silver_weather from Bronze lakehouse (Notebook 1 default)
df_weather = spark.read.format('delta').load(f'{BRONZE}/silver_weather').select(
    'date', 'hour', 'temperature_c', 'precipitation_mm', 'windspeed_kmh'
)

# Data alignment check
taxi_min, taxi_max = df_taxi_hourly.agg(F.min('pickup_date'), F.max('pickup_date')).collect()[0]
w_min, w_max = df_weather.agg(F.min('date'), F.max('date')).collect()[0]
print(f'Taxi data:    {taxi_min} to {taxi_max}')
print(f'Weather data: {w_min} to {w_max}')

# Join on date + hour
df_corr = df_taxi_hourly.join(
    df_weather,
    (df_taxi_hourly.pickup_date == df_weather.date) &
    (df_taxi_hourly.pickup_hour == df_weather.hour),
    'inner'
).select(
    df_taxi_hourly.pickup_date.alias('date'),
    df_taxi_hourly.pickup_hour.alias('hour'),
    'trip_count', 'avg_distance_miles', 'avg_fare_usd', 'avg_duration_min',
    'temperature_c', 'precipitation_mm', 'windspeed_kmh'
)

print(f'\nCorrelation rows: {df_corr.count()}')
df_corr.orderBy('date', 'hour').show(10)

# Save to Gold
df_corr.write.format('delta').mode('overwrite').saveAsTable('gold_taxi_weather_correlation')
print('\ngold_taxi_weather_correlation saved to Gold lakehouse')

# Pearson correlations
corr1 = df_corr.stat.corr('trip_count', 'temperature_c')
corr2 = df_corr.stat.corr('trip_count', 'precipitation_mm')
corr3 = df_corr.stat.corr('trip_count', 'windspeed_kmh')
corr4 = df_corr.stat.corr('avg_fare_usd', 'temperature_c')
print(f'\nPearson: trip_count vs temperature_c    = {corr1:.4f}')
print(f'Pearson: trip_count vs precipitation_mm = {corr2:.4f}')
print(f'Pearson: trip_count vs windspeed_kmh    = {corr3:.4f}')
print(f'Pearson: avg_fare_usd vs temperature_c  = {corr4:.4f}')

StatementMeta(, 33293fc6-eae5-46b7-9578-a38409178160, 75, Finished, Available, Finished, False)

Taxi data:    2024-01-01 to 2024-01-31
Weather data: 2024-01-01 to 2024-01-31

Correlation rows: 744
+----------+----+----------+------------------+------------------+------------------+-------------+----------------+-------------+
|      date|hour|trip_count|avg_distance_miles|      avg_fare_usd|  avg_duration_min|temperature_c|precipitation_mm|windspeed_kmh|
+----------+----+----------+------------------+------------------+------------------+-------------+----------------+-------------+
|2024-01-01|   0|      5375|2.9383274418604755|18.763304186046497|18.027564651162763|          1.9|             0.0|          6.3|
|2024-01-01|   1|      5537|2.8642712660285308|18.184262235867788| 17.79311359942205|          1.8|             0.0|          8.9|
|2024-01-01|   2|      5208| 3.055339861751146|  18.1151017665131|17.222853302611362|          2.8|             0.0|         11.4|
|2024-01-01|   3|      3741|3.1696712109061704|17.669452018176962|15.886105319433321|          2.9|             0

In [47]:
# ============================================================
# INTEGRATION 4b: Great Expectations Validation + Discord Bot
# ============================================================
import great_expectations as gx
from great_expectations.dataset import SparkDFDataset
import requests

SILVER = 'abfss://FabricDataProject@onelake.dfs.fabric.microsoft.com/Silver.Lakehouse/Tables/dbo'
BRONZE = 'abfss://FabricDataProject@onelake.dfs.fabric.microsoft.com/Bronze.Lakehouse/Tables/dbo'

# ---- Validate silver_taxi ----
print('=' * 60)
print('Validating silver_taxi...')
df_taxi = spark.read.format('delta').load(f'{SILVER}/silver_taxi')
ge_taxi = SparkDFDataset(df_taxi)

taxi_results = []
taxi_results.append(('tpep_pickup_datetime not null',
    ge_taxi.expect_column_values_to_not_be_null('tpep_pickup_datetime').success))
taxi_results.append(('trip_distance > 0',
    ge_taxi.expect_column_values_to_be_between('trip_distance', min_value=0.01, max_value=100000).success))
taxi_results.append(('fare_amount in [0.01, 5000]',
    ge_taxi.expect_column_values_to_be_between('fare_amount', min_value=0.01, max_value=5000).success))
taxi_results.append(('row_count in [1M, 5M]',
    ge_taxi.expect_table_row_count_to_be_between(min_value=1_000_000, max_value=5_000_000).success))

print('\n--- silver_taxi ---')
taxi_failures = []
for name, ok in taxi_results:
    print(f'  {"PASS" if ok else "FAIL"}: {name}')
    if not ok: taxi_failures.append(f'silver_taxi: {name}')

# ---- Validate silver_air_quality ----
print('\n' + '=' * 60)
print('Validating silver_air_quality...')
df_aq = spark.read.format('delta').load(f'{SILVER}/silver_air_quality')
ge_aq = SparkDFDataset(df_aq)

aq_results = []
aq_results.append(('pm25_value in [0, 500]',
    ge_aq.expect_column_values_to_be_between('pm25_value', min_value=0, max_value=500).success))
aq_results.append(('datetime not null',
    ge_aq.expect_column_values_to_not_be_null('datetime').success))
aq_results.append(('pm25_value column exists',
    ge_aq.expect_column_to_exist('pm25_value').success))

print('\n--- silver_air_quality ---')
aq_failures = []
for name, ok in aq_results:
    print(f'  {"PASS" if ok else "FAIL"}: {name}')
    if not ok: aq_failures.append(f'silver_air_quality: {name}')

# ---- Validate silver_weather ----
print('\n' + '=' * 60)
print('Validating silver_weather...')
df_w = spark.read.format('delta').load(f'{BRONZE}/silver_weather')
ge_w = SparkDFDataset(df_w)

weather_results = []
weather_results.append(('datetime not null',
    ge_w.expect_column_values_to_not_be_null('datetime').success))
weather_results.append(('temperature_c in [-40, 60]',
    ge_w.expect_column_values_to_be_between('temperature_c', min_value=-40, max_value=60).success))
weather_results.append(('row_count 700-800',
    ge_w.expect_table_row_count_to_be_between(min_value=700, max_value=800).success))

print('\n--- silver_weather ---')
weather_failures = []
for name, ok in weather_results:
    print(f'  {"PASS" if ok else "FAIL"}: {name}')
    if not ok: weather_failures.append(f'silver_weather: {name}')

# ---- BOT TRIGGER ----
print('\n' + '=' * 60)
all_failures = taxi_failures + aq_failures + weather_failures
if all_failures:
    msg = 'Validation failures:\n' + '\n'.join(f'- {f}' for f in all_failures)
    print(f'VALIDATION FAILED: {len(all_failures)} checks failed')
    send_alert('Data Validation FAILED', msg, success=False)
else:
    print('ALL VALIDATIONS PASSED')
    send_alert('Data Validation PASSED', 'All checks passed for silver_taxi, silver_air_quality, silver_weather', success=True)

StatementMeta(, 33293fc6-eae5-46b7-9578-a38409178160, 78, Finished, Available, Finished, False)

Validating silver_taxi...

--- silver_taxi ---
  PASS: tpep_pickup_datetime not null
  PASS: trip_distance > 0
  PASS: fare_amount in [0.01, 5000]
  PASS: row_count in [1M, 5M]

Validating silver_air_quality...

--- silver_air_quality ---
  PASS: pm25_value in [0, 500]
  PASS: datetime not null
  PASS: pm25_value column exists

Validating silver_weather...

--- silver_weather ---
  PASS: datetime not null
  PASS: temperature_c in [-40, 60]
  PASS: row_count 700-800

ALL VALIDATIONS PASSED
Discord webhook: HTTP 204
